## 5 Обучение и выбор модели

**Входы**

- train, validation и правила обработки данных
- Спецификации моделей из этапа 3.5 и критерии оценивания из этапа 1.2

**Выходы**

- Обученная модель, выбранный порог и результаты сравнения

**Что делаем**

Обучим восемь вариантов на train. Выберем модель по AP на validation, затем подберём для неё порог. Test пока не используем

In [ ]:
# dev-only: переходит в корень проекта; в сборку не входит
from pathlib import Path as _NotebookPath
import os as _notebook_os
_notebook_root = next(
    path for path in (_NotebookPath.cwd(), *_NotebookPath.cwd().parents)
    if (path / 'project.json').is_file()
)
_notebook_os.chdir(_notebook_root)


### 5.1 Кандидаты и настройки

Всего попробуем восемь вариантов. В подписях используем короткие имена: Logistic — логистическая регрессия, Forest — лес, Boosting — бустинг

| Модель | Варианты |
| --- | --- |
| Dummy | Один базовый прогноз |
| Логистическая регрессия | C=0.1, C=1 и C=1 с повышенным весом редкого класса |
| Лес | 200 деревьев; минимум 2 или 5 записей в конечной группе дерева, во втором случае — повышенный вес редкого класса |
| Бустинг | 200 шагов; до 15 или 31 конечной группы в каждом дереве |

**C** управляет ограничением коэффициентов логистической модели: меньше C — сильнее ограничение. В подписях **leaf** означает минимальное число записей в конечной группе дерева, **leaves** — максимальное число таких групп. **Balanced** означает, что ошибкам на редком классе при обучении придаётся больший вес

Настройки задаём до сравнения, а не подбираем после просмотра test. Для повторяемости фиксируем случайность числом 42. Каждый вариант обучаем вместе с его обработкой данных на train

In [ ]:
from IPython.display import Markdown
'Метрики и порог: без доступа к test и без переобучения моделей'
import numpy as np
import pandas as pd
from sklearn.metrics import (

    accuracy_score, average_precision_score, balanced_accuracy_score,
    brier_score_loss, f1_score, fbeta_score, precision_recall_curve,
    precision_score, recall_score, roc_auc_score,
)


def classification_metrics(y_true, probabilities, threshold=0.5):
    predictions = np.asarray(probabilities) >= threshold
    return {
        'AP': average_precision_score(y_true, probabilities),
        'ROC-AUC': roc_auc_score(y_true, probabilities),
        'Precision': precision_score(y_true, predictions, zero_division=0),
        'Recall': recall_score(y_true, predictions, zero_division=0),
        'F1': f1_score(y_true, predictions, zero_division=0),
        'F2': fbeta_score(y_true, predictions, beta=2, zero_division=0),
        'Accuracy': accuracy_score(y_true, predictions),
        'Balanced accuracy': balanced_accuracy_score(y_true, predictions),
        'Brier': brier_score_loss(y_true, probabilities),
    }


def select_f2_threshold(y_true, probabilities):
    precision, recall, thresholds = precision_recall_curve(y_true, probabilities)
    p, r = precision[:-1], recall[:-1]
    scores = np.divide(5 * p * r, 4 * p + r, out=np.zeros_like(p), where=(4 * p + r) > 0)
    # При равном F2 выбираем больший порог. Все входы — исключительно validation.
    positions = np.flatnonzero(np.isclose(scores, scores.max(), rtol=0, atol=1e-12))
    best = positions[-1]
    table = pd.DataFrame({'Порог': thresholds, 'Precision': p, 'Recall': r, 'F2': scores})
    return float(thresholds[best]), table


In [ ]:
from sklearn.base import clone
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression

model_candidates = {
    'Dummy: prior': DummyClassifier(strategy='prior'),
    'Logistic: C=0.1': LogisticRegression(C=.1, max_iter=2000, random_state=42),
    'Logistic: C=1': LogisticRegression(C=1, max_iter=2000, random_state=42),
    'Logistic: balanced': LogisticRegression(C=1, class_weight='balanced', max_iter=2000, random_state=42),
    'Forest: leaf=2': RandomForestClassifier(n_estimators=200, min_samples_leaf=2, max_features='sqrt', n_jobs=1, random_state=42),
    'Forest: leaf=5 balanced': RandomForestClassifier(n_estimators=200, min_samples_leaf=5, max_features='sqrt', class_weight='balanced', n_jobs=1, random_state=42),
    'Boosting: leaves=15': HistGradientBoostingClassifier(max_iter=200, max_leaf_nodes=15, learning_rate=.1, l2_regularization=1, early_stopping=False, random_state=42),
    'Boosting: leaves=31': HistGradientBoostingClassifier(max_iter=200, max_leaf_nodes=31, learning_rate=.1, l2_regularization=1, early_stopping=False, random_state=42),
}


### 5.2 Обучение

Каждый кандидат обучаем на train вместе с его обработкой данных. Затем получаем оценки риска на validation

In [ ]:
trained_models = {}
validation_probabilities = {}
comparison_records = []
for _name, _estimator in model_candidates.items():
    _pipeline = Pipeline([('preprocess', clone(preprocessor_template)), ('model', _estimator)])
    _pipeline.fit(X_train, y_train)
    _probability = _pipeline.predict_proba(X_validation)[:, 1]
    trained_models[_name] = _pipeline
    validation_probabilities[_name] = _probability
    comparison_records.append({'Модель': _name, **classification_metrics(y_validation, _probability)})


### 5.3 Выбор модели

Сравниваем кандидатов по AP на validation и выбираем вариант с наибольшим значением

In [ ]:
validation_comparison = pd.DataFrame(comparison_records).set_index('Модель').sort_values('AP', ascending=False, kind='stable')
selected_model_name = str(validation_comparison.index[0])
selected_pipeline = trained_models[selected_model_name]
selected_validation_probability = validation_probabilities[selected_model_name]
display(validation_comparison.round(4))


#### Выбранная модель

Выбираем **градиентный бустинг на деревьях — `Boosting: leaves=31`**: 200 шагов обучения, до 31 конечной группы в каждом дереве. Его AP на validation — **0,9016**, наибольшая среди восьми рассмотренных вариантов

Далее подбираем порог для этого обученного экземпляра. На итоговую оценку передаём его вместе с обученной обработкой признаков; остальные кандидаты остаются результатами сравнения

In [ ]:
selected_estimator_parameters = selected_pipeline.named_steps['model'].get_params()
parameter_labels = {
    'strategy': 'Стратегия', 'C': 'C', 'class_weight': 'Вес классов',
    'n_estimators': 'Число деревьев', 'min_samples_leaf': 'Минимум записей в конечной группе',
    'max_features': 'Признаков на разбиение', 'max_iter': 'Максимум итераций',
    'max_leaf_nodes': 'Максимум конечных групп в дереве', 'learning_rate': 'Шаг обучения',
    'l2_regularization': 'Коэффициент L2', 'early_stopping': 'Досрочное завершение',
    'random_state': 'Начальное состояние генератора',
}
selected_parameters_table = pd.DataFrame([
    {'Настройка': label, 'Выбранное значение': str(selected_estimator_parameters[key])}
    for key, label in parameter_labels.items() if key in selected_estimator_parameters
]).set_index('Настройка')
display(selected_parameters_table)


### 5.4 Выбор порога

Для выбранной модели находим порог с максимальной F2 на validation. При равных значениях берём больший порог

In [ ]:
selected_threshold, threshold_curve = select_f2_threshold(y_validation, selected_validation_probability)
validation_threshold_comparison = pd.DataFrame([
    {'Правило': 'Порог 0.5', 'Порог': .5, **classification_metrics(y_validation, selected_validation_probability, .5)},
    {'Правило': 'Максимум F2 на validation', 'Порог': selected_threshold, **classification_metrics(y_validation, selected_validation_probability, selected_threshold)},
]).set_index('Правило')
display(validation_threshold_comparison.round(4))
# Зафиксированное решение. После этого fit больше не вызывается.
frozen_decision = {
    'model': selected_model_name, 'threshold': selected_threshold, 'selection_metric': 'validation AP',
    'fit_sample': 'train only', 'seed': 42, 'positive_class': 1,
}
display(Markdown(
    f'**Рабочий порог — {selected_threshold:.4f}**. '
    f'F2 на validation — **{validation_threshold_comparison.iloc[1]["F2"]:.4f}**. '
    'Модель, её обработку и порог фиксируем; на этапе 6 проверим это решение на test без изменений'
))


### 5.5 Результат сравнения

Первая диаграмма сравнивает AP кандидатов, вторая показывает зависимость F2, precision и recall от порога

Validation использована и для выбора модели, и для выбора порога, поэтому окончательное качество проверим отдельно на test

In [ ]:
# dev-only: подключает графики сравнения моделей и выбора порога
exec(compile(__import__('pathlib').Path('src/_code/training_plots.py').read_text(encoding='utf-8'), 'src/_code/training_plots.py', 'exec'), globals())
